# Homework: worked answers

Every answer is pinned with an `assert`, so if the data ever changes, this
notebook stops at the first answer that moved. Session 6 answers questions
1 to 10 again in pandas, and these are the numbers it has to match.

In [ ]:
import os
import sqlite3
from pathlib import Path

here = Path.cwd()
while not (here / "data" / "music.db").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

con = sqlite3.connect("data/music.db")


def run(sql, limit=10):
    """Run a query and print the rows as an aligned table."""
    cursor = con.execute(sql)
    headers = [d[0] for d in cursor.description]
    rows = cursor.fetchall()
    widths = [max(len(str(h)), *(len(str(r[i])) for r in rows[:limit] or [[""]]))
              for i, h in enumerate(headers)]
    print("  ".join(str(h).ljust(w) for h, w in zip(headers, widths)))
    print("  ".join("-" * w for w in widths))
    for row in rows[:limit]:
        print("  ".join(str(v).ljust(w) for v, w in zip(row, widths)))
    print(f"({len(rows)} rows)" if len(rows) <= limit
          else f"... {len(rows)} rows in total")


def rows(sql):
    """The raw rows, for checking an answer with assert."""
    return con.execute(sql).fetchall()

# Task 1: ten questions about one table at a time

The two that needed something new were **3** (`IS NULL`, because `= NULL`
finds nothing) and **5** (a year column built out of the date).

## Question 1: how many artists?

In [ ]:
q1 = "SELECT COUNT(*) AS artists FROM artists"
run(q1)
assert rows(q1) == [(40,)]

## Question 2: artists from the Netherlands

In [ ]:
q2 = """
SELECT artist_name, genre
FROM artists
WHERE country = 'Netherlands'
ORDER BY artist_name
"""
run(q2)
assert [r[0] for r in rows(q2)] == ["DJ Kompas", "Neon Harbour", "The Long Commute"]

## Question 3: unknown `formed_year`

One of the two that needs care. `WHERE formed_year = NULL` runs happily and
returns nothing at all, because `NULL` means unknown and an unknown is not
equal to anything, including another unknown.

In [ ]:
run("SELECT artist_name FROM artists WHERE formed_year = NULL")

In [ ]:
q3 = """
SELECT artist_name, country, genre
FROM artists
WHERE formed_year IS NULL
"""
run(q3)
assert sorted(r[0] for r in rows(q3)) == ["Cassette Revival", "Rosewood Lane"]

## Question 4: the five longest plays

In [ ]:
q4 = """
SELECT track_name, played_at, minutes_played
FROM plays
ORDER BY minutes_played DESC
LIMIT 5
"""
run(q4)
assert rows(q4)[0][0] == "Paper Avenue" and rows(q4)[0][2] == 11.01

## Question 5: plays per year

The other one that needs something new. The year is not a column, so you
make one. `strftime('%Y', played_at)` is the date-formatting function, worth
learning because it also gives months and weekdays. `substr(played_at, 1, 4)`
works too, because the dates are stored as `YYYY-MM-DD` text.

In [ ]:
q5 = """
SELECT strftime('%Y', played_at) AS year,
       COUNT(*)                  AS plays
FROM plays
GROUP BY year
ORDER BY year
"""
run(q5)
assert rows(q5) == [("2024", 1122), ("2025", 1061)]

## Question 6: tracks played more than 40 times

Only five tracks clear the bar, out of 168.

In [ ]:
q6 = """
SELECT track_name, COUNT(*) AS plays
FROM plays
GROUP BY track_name
HAVING COUNT(*) > 40
ORDER BY plays DESC
"""
run(q6)
assert rows(q6)[0] == ("Broken Lines", 67) and len(rows(q6)) == 5

## Question 7: minutes per device

In [ ]:
q7 = """
SELECT device,
       ROUND(SUM(minutes_played)) AS total_minutes
FROM plays
GROUP BY device
ORDER BY total_minutes DESC
"""
run(q7)
assert rows(q7)[0] == ("phone", 4308.0)

## Question 8: genres with five or more artists

In [ ]:
q8 = """
SELECT genre, COUNT(*) AS artists
FROM artists
GROUP BY genre
HAVING COUNT(*) >= 5
ORDER BY artists DESC, genre
"""
run(q8)
assert rows(q8)[:2] == [("Electronic", 7), ("Indie", 6)] and len(rows(q8)) == 6

## Question 9: skip rate per device

`COUNT(*)` for all plays and `SUM(skipped)` for the skipped ones, which works
because `skipped` is already a 0 or a 1.

The trap is the division. In SQLite `19 / 179` is `0`, because two whole
numbers give a whole number. `100.0 *` first, with the decimal point, forces
a proper division.

In [ ]:
run("SELECT 19 / 179 AS whole_numbers, ROUND(100.0 * 19 / 179, 1) AS with_decimal")

In [ ]:
q9 = """
SELECT device,
       COUNT(*)      AS plays,
       SUM(skipped)  AS skipped,
       ROUND(100.0 * SUM(skipped) / COUNT(*), 1) AS skipped_pct
FROM plays
GROUP BY device
ORDER BY skipped_pct DESC
"""
run(q9)
assert rows(q9)[0] == ("speaker", 336, 47, 14.0)

## Question 10: the busiest day

Nothing new: `played_at` is one value per day, so it groups like any other
column. But look at the top five, not just the top one.

In [ ]:
q10 = """
SELECT played_at, COUNT(*) AS plays
FROM plays
GROUP BY played_at
ORDER BY plays DESC
LIMIT 5
"""
run(q10)
assert rows(q10)[:2] == [("2024-03-24", 11), ("2024-09-18", 10)]

11 plays against 10 for the runner-up. "The busiest day" sounds like a
finding and is really a coin toss between a handful of ordinary days. Look at
second place before believing a single-row answer.

# Task 2: five questions across tables

The two that need a CTE are **13** and **15**. Both run without one, and
both give a number that is too big.

## Question 11: who gets skipped most?

Without the join this is a list of `artist_id`s nobody can read. Group by
`a.artist_id` rather than the name, so two artists who happened to share a
name would stay apart. The `HAVING` keeps artists with a handful of plays
from topping the list on noise.

In [ ]:
q11 = """
SELECT a.artist_name,
       COUNT(*)       AS plays,
       SUM(p.skipped) AS skips,
       ROUND(100.0 * SUM(p.skipped) / COUNT(*), 1) AS skip_pct
FROM plays p
JOIN artists a USING (artist_id)
GROUP BY a.artist_id
HAVING COUNT(*) >= 30
ORDER BY skip_pct DESC
LIMIT 10
"""
run(q11)
assert rows(q11)[0] == ("Mireille Basset", 40, 8, 20.0)

## Question 12: listening by decade

`formed_year / 10 * 10` uses integer division on purpose: `1996 / 10` is
`199`, times 10 is `1990`. In question 9 it was the trap; here it is the
tool.

In [ ]:
q12 = """
SELECT a.formed_year / 10 * 10        AS decade,
       COUNT(DISTINCT a.artist_id)     AS artists,
       COUNT(*)                        AS plays,
       ROUND(SUM(p.minutes_played), 1) AS minutes
FROM plays p
JOIN artists a USING (artist_id)
GROUP BY decade
ORDER BY decade
"""
run(q12)
assert rows(q12)[0] == (None, 2, 53, 182.8)
assert rows(q12)[3] == (2010, 22, 1087, 4556.9)

Two things to notice:

* **The `None` row** is Rosewood Lane and Cassette Revival, the unknown
  `formed_year` from question 3. `GROUP BY` keeps the unknowns together as
  one group rather than dropping them.
* **The 1990s show 1 artist**, but two were formed then. Iron Fernway (1996)
  has no plays, so the inner join removed him before anything was counted.
  Fine for "how much do you listen", wrong for "how many artists per
  decade".

`COUNT(DISTINCT a.artist_id)` is safe, unlike `SUM(DISTINCT)` in the fan-out
block: `artist_id` is a key, so two equal values really are the same artist.

## Question 13: the award winners' share, per device

**One of the two that needs a CTE.** The obvious query joins `awards`
straight onto `plays`:

In [ ]:
q13_wrong = """
SELECT p.device,
       ROUND(SUM(p.minutes_played), 1) AS total_minutes,
       ROUND(100.0 * SUM(CASE WHEN w.award_id IS NOT NULL
                              THEN p.minutes_played ELSE 0 END)
                   / SUM(p.minutes_played), 1) AS winner_pct
FROM plays p
LEFT JOIN awards w USING (artist_id)
GROUP BY p.device
ORDER BY total_minutes DESC
"""
run(q13_wrong)
assert rows(q13_wrong)[0] == ("phone", 6726.5, 71.8)

**How to tell it is wrong:** question 7 said the phone has 4,308 minutes in
total. Here it has 6,726.5. A total that grows when you join something on is
fan-out. The winners' minutes are inflated far more than everyone else's,
so the percentage is wrong too.

The fix: reduce `awards` to one row per artist **first**, so each play can
match at most once. `LEFT JOIN`, because an inner join to `winners` would
drop every play by an artist with no award.

In [ ]:
q13 = """
WITH winners AS (
    SELECT artist_id, COUNT(*) AS awards
    FROM awards
    GROUP BY artist_id
)
SELECT p.device,
       ROUND(SUM(p.minutes_played), 1) AS total_minutes,
       ROUND(SUM(CASE WHEN w.artist_id IS NOT NULL
                      THEN p.minutes_played ELSE 0 END), 1) AS winner_minutes,
       ROUND(100.0 * SUM(CASE WHEN w.artist_id IS NOT NULL
                              THEN p.minutes_played ELSE 0 END)
                   / SUM(p.minutes_played), 1) AS winner_pct
FROM plays p
LEFT JOIN winners w USING (artist_id)
GROUP BY p.device
ORDER BY total_minutes DESC
"""
run(q13)
assert rows(q13)[0] == ("phone", 4308.2, 2410.6, 56.0)

Phone 4,308.2 minutes: exactly question 7 again. The check is a number from
an independent query, not a number that looks plausible.

## Question 14: inactive artists you still play

In [ ]:
q14 = """
SELECT a.artist_name,
       COUNT(*)                        AS plays,
       ROUND(SUM(p.minutes_played), 1) AS minutes,
       MAX(p.played_at)                AS last_played
FROM plays p
JOIN artists a USING (artist_id)
WHERE a.still_active = 0
GROUP BY a.artist_id
ORDER BY plays DESC
"""
run(q14)
assert rows(q14)[0] == ("The Quiet Wires", 64, 262.5, "2025-12-27")
assert len(rows(q14)) == 4

Two artists are missing, for two different reasons:

* **Gravel Choir** is inactive and has no plays, so the inner join drops it.
  Correct here: the question was who you *still* listen to.
* **Iron Fernway** has `still_active` unknown. `WHERE still_active = 0` does
  not match an unknown, exactly like `= NULL` in question 3.

In [ ]:
run("SELECT still_active, COUNT(*) AS artists FROM artists GROUP BY still_active")

## Question 15: do award winners get listened to more?

**The other one that needs a CTE**, for two reasons at once. It is an average
*of totals* (one total per artist, then the average of those), which needs
two levels of grouping. And `awards` fans the plays out. The obvious attempt:

In [ ]:
q15_wrong = """
SELECT CASE WHEN w.award_id IS NULL THEN 'no award'
            ELSE 'award winner' END    AS grp,
       COUNT(DISTINCT a.artist_id)     AS artists,
       COUNT(*)                        AS rows_after_join,
       ROUND(SUM(p.minutes_played) / COUNT(DISTINCT a.artist_id), 1)
                                       AS minutes_per_artist
FROM artists a
LEFT JOIN plays  p USING (artist_id)
LEFT JOIN awards w USING (artist_id)
GROUP BY grp
"""
run(q15_wrong)
assert rows(q15_wrong)[0] == ("award winner", 17, 2371, 572.5)

2,371 rows: the number from the session's wrong total, turning up again. The
no-award group happens to be right, because those artists have no award rows
to multiply by, which makes the wrong answer even harder to spot.

The fix: one row per artist for plays, one row per artist for awards, then
join. Start `FROM artists` with left joins, and `COALESCE` the two
never-played artists' `NULL` into 0, because the question said to count
every artist.

In [ ]:
q15 = """
WITH per_artist AS (
    SELECT artist_id, SUM(minutes_played) AS minutes
    FROM plays
    GROUP BY artist_id
),
awards_per_artist AS (
    SELECT artist_id, COUNT(*) AS awards
    FROM awards
    GROUP BY artist_id
)
SELECT CASE WHEN aw.artist_id IS NULL THEN 'no award'
            ELSE 'award winner' END            AS grp,
       COUNT(*)                                AS artists,
       ROUND(AVG(COALESCE(s.minutes, 0)), 1)   AS avg_minutes_per_artist
FROM artists a
LEFT JOIN per_artist        s  USING (artist_id)
LEFT JOIN awards_per_artist aw USING (artist_id)
GROUP BY grp
"""
run(q15)
assert rows(q15) == [("award winner", 17, 292.9), ("no award", 23, 173.9)]

Yes: about 120 minutes more per artist for award winners.

# Task 3: an example

**Question:** on average, how many awards does an artist in each genre
have?

In [ ]:
inner = """
SELECT a.genre,
       COUNT(DISTINCT a.artist_id) AS artists,
       ROUND(1.0 * COUNT(w.award_id) / COUNT(DISTINCT a.artist_id), 2)
                                   AS awards_per_artist
FROM artists a
JOIN awards w USING (artist_id)
GROUP BY a.genre
ORDER BY awards_per_artist DESC
"""
run(inner)

In [ ]:
left = inner.replace("JOIN awards", "LEFT JOIN awards")
run(left)
assert len(rows(inner)) == 7 and len(rows(left)) == 8
assert ("Rock", 1, 3.0) in rows(inner) and ("Rock", 5, 0.6) in rows(left)

**The left join answers the question.** It was about an artist in each
genre, and only 17 of the 40 artists have an award. The inner join quietly
changed the question to "how many awards does an award *winner* have": for
Rock that is one artist with 3 awards, so 3.0 became the headline when the
honest answer is 0.6. Afrobeat disappeared altogether, because none of its
artists has won anything, and "0 awards" is an answer, not a missing row.

Neither query is wrong SQL. They answer different questions, and choosing
between them means deciding which question you meant.

In [ ]:
con.close()
print("done")